In [22]:

import argparse
import hashlib
import json
from itertools import product
from pathlib import Path
from itertools import product
import numpy as np
import numbers
import random

### Setup

In [23]:
SKY = 'south'

SINDECS = [-0.5]

NTRIALS = [1000]
FIXED = {
    "signal_injection_gamma": 2.5,
    "dpsi_nbins": 101,
    "weight_field": "oneweight",
    "true_ra_name": "true_ra",
    "true_dec_name": "true_dec",
    "true_energy_name": "true_energy",
    "angular_cutoff_deg": 15.0,
}


SPECTRAL_INDICES = [1.0, 1.7, 2.0, 2.3, 2.5, 2.7, 3.0, 3.3, 4.0]

""" ENERGY_EDGE_CONFIGS = [
[2.0, 2.5, 3., 3.5, 4, 4.5, 5., 6., 10.],
[2.0, 2.5, 3, 3.25, 3.5, 4.25, 5., 6.0, 10.],
[2.0, 2.5, 3., 3.5, 4, 4.5, 5., 5.5, 6., 6.5, 7., 7.5, 8., 10.],
[2.0, 2.25, 2.5, 2.75, 3., 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5., 5.25,  5.5, 5.75, 6., 6.25, 6.5, 6.75, 7., 7.25, 7.5, 7.75, 8., 10.]
] """
ENERGY_EDGE_CONFIGS = [5, 10, 15, 20, 30]
ENERGY_EDGE_CONFIGS_NEW = [50, 75, 100, 200]
if SKY == 'south':
    SINDEC_EDGE_CONFIGS = [
        list(np.linspace(-1, -0.14, 5)),
        list(np.linspace(-1, -0.14, 10)),
        list(np.linspace(-1, -0.14, 15)),
        list(np.linspace(-1, -0.14, 25)), 
        list(np.linspace(-1, -0.14, 45)), 

        ]
    SINDEC_EDGE_CONFIGS_NEW = [
        list(np.linspace(-1, -0.14, 75)), 
        list(np.linspace(-1, -0.14, 100)), 
        list(np.linspace(-1, -0.14, 200)), 
        ]
elif SKY == 'north':
    SINDEC_EDGE_CONFIGS = [
        list(np.linspace(-1, -0.14, 5)),
        list(np.linspace(-0.14, 1, 10)),
        list(np.linspace(-0.14, 1, 15)),
        list(np.linspace(-0.14, 1, 25)), 
        list(np.linspace(-0.14, 1, 45)), 
        ] 
    SINDEC_EDGE_CONFIGS_NEW = [ 
        list(np.linspace(-0.14, 1, 75)), 
        list(np.linspace(-0.14, 1, 100)), 
        list(np.linspace(-0.14, 1, 200)), 
        ] 


SIGMAS = [5, 10, 15, 20, 30, 50]
SIGMAS_NEW = [75, 100]

MINIMUM_COUNTS = [300]

### Defintitions

In [24]:
def canonicalize_edges(edges, precision=8):
    return [round(float(x), precision) for x in edges]

def short_hash(values, digits=6):
    values = canonicalize_edges(values)
    payload = json.dumps(values, separators=(",", ":"), sort_keys=True)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:digits]


def make_candidate_id(log10energy, dec_sindec_edges, sigma, minimum_counts):
    if isinstance(log10energy, numbers.Integral):
        energy_part = f"E{log10energy}"
    else:
        energy_part = f"E{len(log10energy)-1}_{short_hash(log10energy)}"

    if isinstance(dec_sindec_edges, numbers.Integral):
        sindec_part = f"D{dec_sindec_edges}"
    else:
        sindec_part = f"D{len(dec_sindec_edges)-1}_{short_hash(dec_sindec_edges)}"

    return (
        f"{energy_part}_"
        f"{sindec_part}_"
        f"S{int(sigma):02d}_MC{int(minimum_counts)}"
    )
    
def make_candidate(log10energy, dec_sindec_edges, sigma, minimum_counts):
    return {
        "id": make_candidate_id(
            log10energy=log10energy,
            dec_sindec_edges=dec_sindec_edges,
            sigma=sigma,
            minimum_counts=minimum_counts,
        ),
        "minimum_counts": int(minimum_counts),
        "parametrization_bins": {
            "log10energy": log10energy,
            "dec_sindec_edges": dec_sindec_edges,
            "sigma": int(sigma),
        },
    }


def _check_unique_ids(candidates):
    ids = [candidate["id"] for candidate in candidates]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate candidate IDs detected.")


def _homogeneous_downsample(combinations, target_size, seed=0):
    """
    Deterministically samples the full parameter-space list as evenly as possible.

    This assumes `combinations` is ordered Cartesian-product output.
    It picks evenly spaced indices across the full list and optionally jitters
    within each interval using `seed`.
    """
    n_total = len(combinations)

    if target_size is None or target_size >= n_total:
        return combinations

    if target_size <= 0:
        raise ValueError("target_size must be positive.")

    rng = random.Random(seed)

    sampled = []
    used_indices = set()

    for i in range(target_size):
        start = i * n_total / target_size
        end = (i + 1) * n_total / target_size

        idx_min = int(start)
        idx_max = max(idx_min, int(end) - 1)

        idx = rng.randint(idx_min, idx_max)

        while idx in used_indices:
            idx = (idx + 1) % n_total

        used_indices.add(idx)
        sampled.append(combinations[idx])

    return sampled


def build_all_combinations(target_size=None, seed=0):
    combinations = list(
        product(
            ENERGY_EDGE_CONFIGS,
            SINDEC_EDGE_CONFIGS,
            SIGMAS,
            MINIMUM_COUNTS,
        )
    )

    combinations = _homogeneous_downsample(
        combinations=combinations,
        target_size=target_size,
        seed=seed,
    )

    candidates = [
        make_candidate(
            log10energy=log10energy,
            dec_sindec_edges=dec_sindec_edges,
            sigma=sigma,
            minimum_counts=minimum_counts,
        )
        for log10energy, dec_sindec_edges, sigma, minimum_counts in combinations
    ]

    _check_unique_ids(candidates)

    return candidates

def build_new_combinations(target_size=None, seed=0):
    all_energy = ENERGY_EDGE_CONFIGS + ENERGY_EDGE_CONFIGS_NEW
    all_sindec = SINDEC_EDGE_CONFIGS + SINDEC_EDGE_CONFIGS_NEW
    all_sigmas = SIGMAS + SIGMAS_NEW

    combinations = [
        combo
        for combo in product(
            all_energy,
            all_sindec,
            all_sigmas,
            MINIMUM_COUNTS,
        )
        if not (
            combo[0] in ENERGY_EDGE_CONFIGS
            and combo[1] in SINDEC_EDGE_CONFIGS
            and combo[2] in SIGMAS
        )
    ]

    combinations = _homogeneous_downsample(
        combinations=combinations,
        target_size=target_size,
        seed=seed,
    )

    candidates = [
        make_candidate(
            log10energy=log10energy,
            dec_sindec_edges=dec_sindec_edges,
            sigma=sigma,
            minimum_counts=minimum_counts,
        )
        for log10energy, dec_sindec_edges, sigma, minimum_counts in combinations
    ]

    _check_unique_ids(candidates)

    return candidates

### Execution

for `build_all_combinations` one can choose a target size of configurations if desired, which then applies homogenous downsampling to the dataset

In [25]:
OUTPUT_FILE = SKY + "_candidates.json"
INDENT = 2
candidates = build_new_combinations( seed = 0)
print(f'Number of candidates: {len(candidates)}')
print(f'Full scan would be: {len(build_new_combinations(seed = 0))}')
config = {
    "spectral_indices": SPECTRAL_INDICES,
    "fixed": FIXED,
    "candidates": candidates,
}


Number of candidates: 426
Full scan would be: 426


In [26]:

with open(OUTPUT_FILE, "w") as f:
    json.dump(config, f, indent=INDENT)

print(f"Wrote {len(config['candidates'])} candidates to {OUTPUT_FILE}")
config["candidates"][:3]

Wrote 426 candidates to south_candidates.json


[{'id': 'E5_D4_902b96_S75_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': 5,
   'dec_sindec_edges': [np.float64(-1.0),
    np.float64(-0.785),
    np.float64(-0.5700000000000001),
    np.float64(-0.355),
    np.float64(-0.14)],
   'sigma': 75}},
 {'id': 'E5_D4_902b96_S100_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': 5,
   'dec_sindec_edges': [np.float64(-1.0),
    np.float64(-0.785),
    np.float64(-0.5700000000000001),
    np.float64(-0.355),
    np.float64(-0.14)],
   'sigma': 100}},
 {'id': 'E5_D9_fc2efc_S75_MC300',
  'minimum_counts': 300,
  'parametrization_bins': {'log10energy': 5,
   'dec_sindec_edges': [np.float64(-1.0),
    np.float64(-0.9044444444444444),
    np.float64(-0.8088888888888889),
    np.float64(-0.7133333333333334),
    np.float64(-0.6177777777777778),
    np.float64(-0.5222222222222221),
    np.float64(-0.42666666666666664),
    np.float64(-0.33111111111111113),
    np.float64(-0.23555555555555552),
    np.

### Create corresponding list for bkg trials submission file:

In [27]:

SKYS = [str(SKY)]
CANDIDATE_IDS = [c["id"] for c in config["candidates"]]

CANDIDATE_IDS = list(set(CANDIDATE_IDS))

print("queue SINDEC, NTRIALS, CANDIDATE_ID, SKY from (")

for sindec, ntrials, candidate_id, sky in product(
    SINDECS,
    NTRIALS,
    CANDIDATE_IDS,
    SKYS
):
    print(f"{sindec} {ntrials} {candidate_id} {sky}")

print(")")

queue SINDEC, NTRIALS, CANDIDATE_ID, SKY from (
-0.5 1000 E5_D199_3d5b24_S50_MC300 south
-0.5 1000 E20_D99_6da66e_S100_MC300 south
-0.5 1000 E75_D4_902b96_S75_MC300 south
-0.5 1000 E15_D4_902b96_S75_MC300 south
-0.5 1000 E200_D14_214963_S50_MC300 south
-0.5 1000 E15_D74_23839b_S10_MC300 south
-0.5 1000 E20_D14_214963_S75_MC300 south
-0.5 1000 E200_D4_902b96_S30_MC300 south
-0.5 1000 E75_D74_23839b_S15_MC300 south
-0.5 1000 E15_D74_23839b_S15_MC300 south
-0.5 1000 E75_D14_214963_S75_MC300 south
-0.5 1000 E200_D199_3d5b24_S100_MC300 south
-0.5 1000 E75_D99_6da66e_S10_MC300 south
-0.5 1000 E20_D74_23839b_S05_MC300 south
-0.5 1000 E30_D199_3d5b24_S75_MC300 south
-0.5 1000 E5_D99_6da66e_S10_MC300 south
-0.5 1000 E30_D24_b2448a_S100_MC300 south
-0.5 1000 E5_D199_3d5b24_S30_MC300 south
-0.5 1000 E50_D44_927d0d_S10_MC300 south
-0.5 1000 E100_D44_927d0d_S20_MC300 south
-0.5 1000 E20_D4_902b96_S100_MC300 south
-0.5 1000 E20_D4_902b96_S75_MC300 south
-0.5 1000 E200_D74_23839b_S30_MC300 south
-0.5

In [28]:
num_candidates_tot = len(SINDECS)* len(NTRIALS)* len(CANDIDATE_IDS)
print(f'#Candidates (total): {num_candidates_tot}')
time_per_candidate_hours = 0.3 
print(f'Est. runtime @32cpu (h): {num_candidates_tot* time_per_candidate_hours}') 
print(f'Est. runtime @32cpu (d): {num_candidates_tot* time_per_candidate_hours /24}') 

#Candidates (total): 426
Est. runtime @32cpu (h): 127.8
Est. runtime @32cpu (d): 5.325


In [29]:
import os
for c in CANDIDATE_IDS: 
    os.makedirs(f'/data/user/fkrafft/split_sky_trials/{SKY}/{c}/log', exist_ok= True)

In [30]:
len(CANDIDATE_IDS)

426

In [31]:
np.r_[:31:3]

array([ 0,  3,  6,  9, 12, 15, 18, 21, 24, 27, 30])